# Curation format

In this tutorial we will use the spikeinterface curation format, to save some remove/merge/split/labeling attached to an analyzer
and then check then using the spikeinterface-gui.


In [1]:
import json
import warnings
from pathlib import Path

import spikeinterface.full as si


In [2]:
# change this path to point to the folder your data is in
# base_folder = Path("/home/samuel/Documents/DataSpikeSorting/school_neuralnet_2026/")
base_folder = Path("/mnt/data/sam/DataSpikeSorting/school_neuralnet_2026/")
# base_folder = Path("/home/pierre/school_neuralnet_2026/")

data_folder = base_folder / "data"
name = "sub-A3702_ses-191126-3600s"
result_folder = base_folder / "results" / name
result_folder.mkdir(exist_ok=True)



In [3]:
sorter_name = "lupin"
# sorter_name = "kilosort4"
# sorter_name = "dartsort"

analyzer_folder = result_folder / f"analyzer_{sorter_name}_with_labels"
analyzer = si.load(analyzer_folder)
print(analyzer)


SortingAnalyzer: 64 channels - 164 units - 1 segments - binary_folder - sparse - has recording
Loaded 11 extensions: correlograms, template_metrics, template_similarity, quality_metrics, spike_amplitudes, unit_locations, waveforms, spike_locations, templates, random_spikes, noise_levels


In [4]:
# find some potential merges

potential_merges = si.compute_merge_unit_groups(
    analyzer,
    preset="slay",
    steps_params={"template_similarity": {"template_diff_thresh": 0.5}}
)

potential_merges

[[np.int64(8), np.int64(86)],
 [np.int64(70), np.int64(75)],
 [np.int64(94), np.int64(97)],
 [np.int64(191), np.int64(243)],
 [np.int64(215), np.int64(216)]]

In [5]:
# we will use unitrefine labels as pre-computed "quality"
unitrefine_labels = analyzer.get_sorting_property("unitrefine_label").copy()
unitrefine_labels[unitrefine_labels == "sua"] = "good"
unitrefine_labels[unitrefine_labels == "mua"] = "MUA"

label_definitions = {
    "quality": dict(name="quality", label_options=["good", "MUA", "noise"], exclusive=True),
}
manual_labels = [
    {"unit_id": unit_id, "labels": {"quality": [unitrefine_labels[unit_index]]}}
    for unit_index, unit_id in enumerate(analyzer.unit_ids)
]

In [6]:
curation_dict = dict(
    format_version="2",
    unit_ids=analyzer.unit_ids,
    label_definitions=label_definitions, 
    merges=[{"unit_ids": p} for p in potential_merges],
    removed=analyzer.unit_ids[unitrefine_labels == "noise"],
    manual_labels=manual_labels
)

si.validate_curation_dict(curation_dict)

In [7]:
curation_dict

{'format_version': '2',
 'unit_ids': array([  4,   5,   8,  12,  19,  26,  27,  30,  31,  34,  44,  51,  59,
         62,  63,  64,  65,  67,  68,  69,  70,  71,  72,  73,  74,  75,
         76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  88,  89,
         90,  91,  92,  93,  94,  95,  97,  99, 100, 101, 103, 105, 106,
        107, 108, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121,
        122, 123, 124, 126, 127, 128, 129, 130, 131, 132, 133, 135, 136,
        137, 139, 140, 141, 142, 143, 146, 148, 150, 151, 153, 154, 155,
        156, 157, 159, 160, 162, 163, 164, 165, 166, 167, 168, 169, 170,
        173, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184, 185, 186,
        187, 188, 189, 190, 191, 194, 195, 197, 200, 203, 204, 205, 206,
        207, 208, 209, 211, 212, 215, 216, 217, 218, 219, 220, 221, 222,
        224, 226, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 239,
        241, 242, 243, 246, 247, 248, 249, 250]),
 'label_definitions': {'quality': {'na

In [8]:
from spikeinterface_gui import run_mainwindow

run_mainwindow(
    analyzer,
    displayed_unit_properties=["threshold_label", "unitrefine_label", "bombcell_label"],
    curation=True,
    curation_dict=curation_dict,
    mode="desktop"
)


: 

In [ ]:
with open(result_folder / "refined_curation.json") as f:
    refined_curation_dict = json.load(f)

In [ ]:
analyzer_curated = si.apply_curation(analyzer, refined_curation_dict, sparsity_overlap=0.5)
analyzer_curated